# Standalone SegFormer benchmark notebook

This notebook is fully standalone for the **pre-trained benchmark**.

It includes:
1. Local dataset loading for `background`, `person`, `car`, `dog`
2. Balanced training with augmentation
3. Fine-tuning a pre-trained **SegFormer**
4. Local validation/test metrics
5. Evaluation on 100 unseen Open Images images
6. Saving and reloading the fine-tuned model for presentation


In [6]:
# Run if needed:
# %pip install torch torchvision numpy pillow matplotlib tqdm transformers accelerate fiftyone
# %pip install transformers


In [7]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Tuple

import numpy as np
import torch
from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

FAST_MODE = True
CACHE_DATA = True
TARGET_PER_CLASS = 400
TEST_TOTAL = 100
VAL_FRACTION = 0.15
SEED = 42

CLASS_NAMES = ["background", "person", "car", "dog"]
NUM_CLASSES = len(CLASS_NAMES)
ID_PERSON, ID_CAR, ID_DOG = 1, 2, 3

CPU_CORES = os.cpu_count() or 8
IMAGE_SIZE = (128, 128) if FAST_MODE else (256, 256)
BATCH_SIZE = 4 if device.type == "cuda" else 2
NUM_WORKERS = 2 if device.type == "cuda" else 0
CHECKPOINT_DIR = Path.cwd() / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
else:
    torch.set_num_threads(min(8, CPU_CORES))
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

def seed_everything(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

@dataclass(frozen=True)
class SegAugmentConfig:
    hflip_p: float = 0.5
    affine_p: float = 0.35
    degrees: float = 12.0
    translate: float = 0.08
    scale: Tuple[float, float] = (0.9, 1.1)
    brightness: Tuple[float, float] = (0.85, 1.15)
    contrast: Tuple[float, float] = (0.85, 1.15)
    saturation: Tuple[float, float] = (0.85, 1.15)

class CachedBinarySegFolderDataset(Dataset):
    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        *,
        foreground_class_id: int,
        size: Tuple[int, int],
        cache: bool = True,
        image_exts: Sequence[str] = (".jpg", ".jpeg", ".png", ".webp"),
        mask_exts: Sequence[str] = (".png", ".jpg", ".jpeg", ".gif"),
        mask_prefix: str = "",
        mask_suffix: str = "",
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.foreground_class_id = int(foreground_class_id)
        self.size = tuple(size)
        self.cache = bool(cache)
        self.image_exts = tuple(e.lower() for e in image_exts)
        self.mask_exts = tuple(e.lower() for e in mask_exts)
        self.mask_prefix = str(mask_prefix)
        self.mask_suffix = str(mask_suffix)

        images = [
            p
            for p in sorted(self.images_dir.iterdir())
            if p.is_file() and p.suffix.lower() in self.image_exts
        ]
        if not images:
            raise FileNotFoundError(f"No images found in {self.images_dir}")

        self.pairs = []
        for img_path in images:
            found = None
            for ext in self.mask_exts:
                candidate = self.masks_dir / f"{self.mask_prefix}{img_path.stem}{self.mask_suffix}{ext}"
                if candidate.exists():
                    found = candidate
                    break
            if found is not None:
                self.pairs.append((img_path, found))

        if not self.pairs:
            raise FileNotFoundError(f"No matching masks found in {self.masks_dir}")

        self.samples = None
        if self.cache:
            self.samples = [self._load_pair(img_path, mask_path) for img_path, mask_path in self.pairs]

    def _load_pair(self, img_path: Path, mask_path: Path):
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        img = TF.resize(img, self.size, interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.size, interpolation=InterpolationMode.NEAREST)

        img_t = TF.to_tensor(img)
        mask_np = np.array(mask, dtype=np.uint8)
        out = np.zeros_like(mask_np, dtype=np.uint8)
        out[mask_np > 0] = np.uint8(self.foreground_class_id)
        mask_t = torch.from_numpy(out)
        return img_t.contiguous(), mask_t.contiguous()

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        if self.samples is None:
            img_t, mask_t = self._load_pair(*self.pairs[idx])
        else:
            img_t, mask_t = self.samples[idx]
        return img_t.clone(), mask_t.clone()

class AugmentedDataset(Dataset):
    def __init__(self, base: Dataset, cfg: Optional[SegAugmentConfig] = None):
        self.base = base
        self.cfg = cfg

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, mask = self.base[idx]
        if self.cfg is None:
            return img, mask

        if torch.rand(1).item() < self.cfg.hflip_p:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[1])

        if torch.rand(1).item() < self.cfg.affine_p:
            max_dx = self.cfg.translate * img.shape[2]
            max_dy = self.cfg.translate * img.shape[1]
            translate = [
                int(round(torch.empty(1).uniform_(-max_dx, max_dx).item())),
                int(round(torch.empty(1).uniform_(-max_dy, max_dy).item())),
            ]
            angle = float(torch.empty(1).uniform_(-self.cfg.degrees, self.cfg.degrees).item())
            scale = float(torch.empty(1).uniform_(*self.cfg.scale).item())

            img = TF.affine(
                img,
                angle=angle,
                translate=translate,
                scale=scale,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            mask = TF.affine(
                mask.unsqueeze(0).float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).squeeze(0).to(torch.uint8)

        b = float(torch.empty(1).uniform_(*self.cfg.brightness).item())
        c = float(torch.empty(1).uniform_(*self.cfg.contrast).item())
        s = float(torch.empty(1).uniform_(*self.cfg.saturation).item())
        img = TF.adjust_brightness(img, b)
        img = TF.adjust_contrast(img, c)
        img = TF.adjust_saturation(img, s)
        return img.clamp(0.0, 1.0).contiguous(), mask.contiguous()

class OversampledDataset(Dataset):
    def __init__(self, base: Dataset, length: int, seed: int):
        self.base = base
        self.length = int(length)
        if len(base) == 0:
            raise ValueError("Base dataset is empty")
        g = torch.Generator().manual_seed(int(seed))
        self.indices = torch.randint(0, len(base), (self.length,), generator=g).tolist()

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return self.base[self.indices[idx]]

def split_train_val_test(n: int, n_test: int, val_fraction: float, seed: int):
    if n_test >= n:
        raise ValueError(f"n_test must be < n. Got n_test={n_test}, n={n}")
    g = torch.Generator().manual_seed(int(seed))
    perm = torch.randperm(n, generator=g).tolist()
    test_idx = perm[:n_test]
    remaining = perm[n_test:]
    n_val = max(1, int(round(len(remaining) * val_fraction)))
    n_val = min(n_val, len(remaining) - 1)
    val_idx = remaining[:n_val]
    train_idx = remaining[n_val:]
    return train_idx, val_idx, test_idx

DATA_ROOT = Path.cwd() / "data"
AUG_CFG = SegAugmentConfig()

car_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "car" / "images",
    DATA_ROOT / "car" / "masks",
    foreground_class_id=ID_CAR,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
)
person_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "person" / "images",
    DATA_ROOT / "person" / "masks",
    foreground_class_id=ID_PERSON,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
)
dog_base = CachedBinarySegFolderDataset(
    DATA_ROOT / "dog" / "images",
    DATA_ROOT / "dog" / "masks",
    foreground_class_id=ID_DOG,
    size=IMAGE_SIZE,
    cache=CACHE_DATA,
    mask_prefix="annotated_",
)

base_sizes = {"car": len(car_base), "person": len(person_base), "dog": len(dog_base)}
print("Base sizes:", base_sizes)

test_base = TEST_TOTAL // 3
remainder = TEST_TOTAL - 3 * test_base
test_counts = {"car": test_base, "person": test_base, "dog": test_base}
for key in ("car", "person", "dog")[:remainder]:
    test_counts[key] += 1
for key in list(test_counts.keys()):
    test_counts[key] = min(test_counts[key], base_sizes[key] - 2)

car_train_idx, car_val_idx, car_test_idx = split_train_val_test(base_sizes["car"], test_counts["car"], VAL_FRACTION, SEED)
person_train_idx, person_val_idx, person_test_idx = split_train_val_test(base_sizes["person"], test_counts["person"], VAL_FRACTION, SEED + 1)
dog_train_idx, dog_val_idx, dog_test_idx = split_train_val_test(base_sizes["dog"], test_counts["dog"], VAL_FRACTION, SEED + 2)

train_ds = ConcatDataset([
    OversampledDataset(AugmentedDataset(Subset(car_base, car_train_idx), AUG_CFG), TARGET_PER_CLASS, seed=SEED + 10),
    OversampledDataset(AugmentedDataset(Subset(person_base, person_train_idx), AUG_CFG), TARGET_PER_CLASS, seed=SEED + 20),
    OversampledDataset(AugmentedDataset(Subset(dog_base, dog_train_idx), AUG_CFG), TARGET_PER_CLASS, seed=SEED + 30),
])
val_ds = ConcatDataset([
    Subset(car_base, car_val_idx),
    Subset(person_base, person_val_idx),
    Subset(dog_base, dog_val_idx),
])
test_ds = ConcatDataset([
    Subset(car_base, car_test_idx),
    Subset(person_base, person_test_idx),
    Subset(dog_base, dog_test_idx),
])

print("Train size:", len(train_ds))
print("Val size  :", len(val_ds))
print("Test size :", len(test_ds))


Using device: cpu
Base sizes: {'car': 256, 'person': 180, 'dog': 180}
Train size: 1200
Val size  : 77
Test size : 100


In [8]:
import torch.nn as nn
import torch.nn.functional as F

@torch.no_grad()
def estimate_class_weights(dataset, num_classes: int):
    counts = torch.zeros(num_classes, dtype=torch.float64)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    for _, yb in loader:
        counts += torch.bincount(yb.view(-1), minlength=num_classes).to(torch.float64)
    weights = counts.sum() / (num_classes * counts.clamp_min(1.0))
    weights = weights / weights.mean()
    return weights.to(device=device, dtype=torch.float32)

class SegmentationLoss(nn.Module):
    def __init__(self, ce_weight: torch.Tensor, dice_weight: float = 0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=ce_weight)
        self.dice_weight = float(dice_weight)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        ce = self.ce(logits, targets)
        probs = torch.softmax(logits, dim=1)
        target_1h = F.one_hot(targets, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()

        probs_fg = probs[:, 1:]
        target_fg = target_1h[:, 1:]
        dims = (0, 2, 3)
        intersection = (probs_fg * target_fg).sum(dim=dims)
        denominator = probs_fg.sum(dim=dims) + target_fg.sum(dim=dims)
        dice = (2.0 * intersection + 1e-6) / (denominator + 1e-6)
        dice_loss = 1.0 - dice.mean()
        return ce + self.dice_weight * dice_loss


In [9]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

SEGFORMER_MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"
SEGFORMER_EPOCHS = 10
SEGFORMER_LR = 6e-5
SEGFORMER_WEIGHT_DECAY = 1e-4
SEGFORMER_BATCH_SIZE = BATCH_SIZE
SEGFORMER_SAVE_DIR = CHECKPOINT_DIR / "segformer_standalone"
SEGFORMER_SAVE_DIR.mkdir(parents=True, exist_ok=True)

id2label = {i: name for i, name in enumerate(CLASS_NAMES)}
label2id = {name: i for i, name in id2label.items()}

segformer_processor = SegformerImageProcessor(
    do_resize=True,
    size={"height": IMAGE_SIZE[0], "width": IMAGE_SIZE[1]},
    do_reduce_labels=False,
)

def segformer_collate(batch):
    images, masks = zip(*batch)
    images_np = [(img.permute(1, 2, 0).numpy() * 255).astype(np.uint8) for img in images]
    masks_np = [mask.numpy().astype(np.uint8) for mask in masks]
    return segformer_processor(
        images=images_np,
        segmentation_maps=masks_np,
        return_tensors="pt",
    )

loader_kwargs = dict(
    batch_size=SEGFORMER_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    collate_fn=segformer_collate,
)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

segformer_train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
segformer_val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
segformer_test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

segformer_model = SegformerForSemanticSegmentation.from_pretrained(
    SEGFORMER_MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
).to(device)

segformer_optimizer = torch.optim.AdamW(
    segformer_model.parameters(),
    lr=SEGFORMER_LR,
    weight_decay=SEGFORMER_WEIGHT_DECAY,
)
segformer_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    segformer_optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)
segformer_class_weights = estimate_class_weights(train_ds, NUM_CLASSES)
segformer_criterion = SegmentationLoss(segformer_class_weights, dice_weight=0.5)
segformer_scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

print("SegFormer ready:", SEGFORMER_MODEL_NAME)


c:\Users\kipra\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kipra\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kipra\.cache\huggingface\hub\models--nvidia--segformer-b0-finetuned-ade-512-512. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to

SegFormer ready: nvidia/segformer-b0-finetuned-ade-512-512


In [10]:
@torch.no_grad()
def evaluate_segformer(model, loader, num_classes: int):
    model.eval()
    total_loss = 0.0
    conf = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)

    for batch in loader:
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, dtype=torch.long, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(pixel_values=pixel_values)
            logits = F.interpolate(
                outputs.logits,
                size=labels.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            loss = segformer_criterion(logits, labels)

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        idx = num_classes * labels.view(-1) + preds.view(-1)
        conf += torch.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)

    conf = conf.float()
    diag = conf.diag()
    recall = diag / (conf.sum(1) + 1e-7)
    precision = diag / (conf.sum(0) + 1e-7)
    f1 = 2.0 * precision * recall / (precision + recall + 1e-7)
    denom = conf.sum(1) + conf.sum(0) - diag + 1e-7
    iou = diag / denom
    pixel_acc = (diag.sum() / (conf.sum() + 1e-7)).item()

    return {
        "loss": total_loss / max(1, len(loader)),
        "pixel_acc": pixel_acc,
        "miou_fg": float(iou[1:].mean().item()),
        "iou_per_class": iou.detach().cpu().numpy(),
        "precision_per_class": precision.detach().cpu().numpy(),
        "recall_per_class": recall.detach().cpu().numpy(),
        "f1_per_class": f1.detach().cpu().numpy(),
    }

def train_one_epoch_segformer(model, loader):
    model.train()
    running_loss = 0.0

    for batch in loader:
        pixel_values = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, dtype=torch.long, non_blocking=True)

        segformer_optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(pixel_values=pixel_values)
            logits = F.interpolate(
                outputs.logits,
                size=labels.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            loss = segformer_criterion(logits, labels)

        if device.type == "cuda":
            segformer_scaler.scale(loss).backward()
            segformer_scaler.step(segformer_optimizer)
            segformer_scaler.update()
        else:
            loss.backward()
            segformer_optimizer.step()

        running_loss += loss.item()

    return running_loss / max(1, len(loader))

best_val_miou = -float("inf")
best_epoch = 0
best_state = None

for epoch in range(1, SEGFORMER_EPOCHS + 1):
    train_loss = train_one_epoch_segformer(segformer_model, segformer_train_loader)
    val_metrics = evaluate_segformer(segformer_model, segformer_val_loader, NUM_CLASSES)
    segformer_scheduler.step(val_metrics["miou_fg"])

    if val_metrics["miou_fg"] > best_val_miou:
        best_val_miou = val_metrics["miou_fg"]
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in segformer_model.state_dict().items()
        }

    print(
        f"Epoch {epoch:02d}/{SEGFORMER_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"pixAcc={val_metrics['pixel_acc']:.3f} | "
        f"mIoU_fg={val_metrics['miou_fg']:.3f}"
    )

if best_state is not None:
    segformer_model.load_state_dict(best_state)

segformer_model.save_pretrained(SEGFORMER_SAVE_DIR)
segformer_processor.save_pretrained(SEGFORMER_SAVE_DIR)
print("Saved fine-tuned SegFormer to:", SEGFORMER_SAVE_DIR)
print("Best epoch:", best_epoch)

test_metrics = evaluate_segformer(segformer_model, segformer_test_loader, NUM_CLASSES)
print(
    f"SEGFORMER TEST | loss={test_metrics['loss']:.4f} | "
    f"pixAcc={test_metrics['pixel_acc']:.3f} | "
    f"mIoU_fg={test_metrics['miou_fg']:.3f}"
)
for idx, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:>10} | "
        f"IoU={test_metrics['iou_per_class'][idx]:.3f} | "
        f"Precision={test_metrics['precision_per_class'][idx]:.3f} | "
        f"Recall={test_metrics['recall_per_class'][idx]:.3f} | "
        f"F1={test_metrics['f1_per_class'][idx]:.3f}"
    )


Epoch 01/10 | train_loss=1.0117 | val_loss=0.6282 | pixAcc=0.847 | mIoU_fg=0.655
Epoch 02/10 | train_loss=0.6621 | val_loss=0.7099 | pixAcc=0.828 | mIoU_fg=0.620
Epoch 03/10 | train_loss=0.5463 | val_loss=0.6250 | pixAcc=0.863 | mIoU_fg=0.679
Epoch 04/10 | train_loss=0.5046 | val_loss=0.6041 | pixAcc=0.881 | mIoU_fg=0.722
Epoch 05/10 | train_loss=0.4571 | val_loss=0.5412 | pixAcc=0.887 | mIoU_fg=0.744
Epoch 06/10 | train_loss=0.4808 | val_loss=0.6002 | pixAcc=0.891 | mIoU_fg=0.731
Epoch 07/10 | train_loss=0.4034 | val_loss=0.6013 | pixAcc=0.895 | mIoU_fg=0.737
Epoch 08/10 | train_loss=0.4134 | val_loss=0.6158 | pixAcc=0.902 | mIoU_fg=0.752
Epoch 09/10 | train_loss=0.3823 | val_loss=0.6577 | pixAcc=0.894 | mIoU_fg=0.744
Epoch 10/10 | train_loss=0.3767 | val_loss=0.5419 | pixAcc=0.906 | mIoU_fg=0.763


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 26.72it/s]

Saved fine-tuned SegFormer to: c:\Users\kipra\OneDrive\Dokumente\New project\checkpoints\segformer_standalone
Best epoch: 10


SEGFORMER TEST | loss=0.6863 | pixAcc=0.892 | mIoU_fg=0.722
background | IoU=0.869 | Precision=0.975 | Recall=0.889 | F1=0.930
    person | IoU=0.655 | Precision=0.799 | Recall=0.785 | F1=0.792
       car | IoU=0.854 | Precision=0.855 | Recall=0.999 | F1=0.921
       dog | IoU=0.655 | Precision=0.687 | Recall=0.935 | F1=0.792


In [11]:
# Evaluation on 100 unseen Open Images validation images
import fiftyone.zoo as foz
from fiftyone.utils.labels import objects_to_segmentations

OI_CLASSES = ["Person", "Car", "Dog"]
OI_MASK_TARGETS = {1: "Person", 2: "Car", 3: "Dog"}
OI_MAX_SAMPLES = 100

oi_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["segmentations"],
    classes=OI_CLASSES,
    only_matching=True,
    max_samples=OI_MAX_SAMPLES,
    shuffle=True,
    seed=SEED,
    dataset_name="open-images-v7-person-car-dog-seg-100",
)

instance_field = None
for candidate in ("ground_truth", "detections", "segmentations"):
    try:
        objects_to_segmentations(
            oi_dataset,
            candidate,
            "gt_semantic",
            mask_targets=OI_MASK_TARGETS,
            overwrite=True,
            progress=True,
        )
        instance_field = candidate
        break
    except Exception:
        continue

if instance_field is None:
    raise RuntimeError("Could not find the Open Images instance-segmentation field")

class OpenImagesSemanticDataset(Dataset):
    def __init__(self, samples, size):
        self.samples = list(samples.iter_samples(progress=True))
        self.size = tuple(size)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = Image.open(sample.filepath).convert("RGB")
        img = TF.resize(img, self.size, interpolation=InterpolationMode.BILINEAR)
        img_t = TF.to_tensor(img)

        seg = sample["gt_semantic"]
        mask_np = seg.get_mask()
        mask_t = torch.from_numpy(mask_np.astype("uint8"))
        mask_t = TF.resize(
            mask_t.unsqueeze(0).float(),
            self.size,
            interpolation=InterpolationMode.NEAREST,
        ).squeeze(0).to(torch.long)
        return img_t.contiguous(), mask_t.contiguous()

oi_ds = OpenImagesSemanticDataset(oi_dataset, IMAGE_SIZE)
oi_loader = DataLoader(
    oi_ds,
    batch_size=SEGFORMER_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    collate_fn=segformer_collate,
)

oi_metrics = evaluate_segformer(segformer_model, oi_loader, NUM_CLASSES)
print(
    f"SEGFORMER OPENIMAGES-100 | loss={oi_metrics['loss']:.4f} | "
    f"pixAcc={oi_metrics['pixel_acc']:.3f} | "
    f"mIoU_fg={oi_metrics['miou_fg']:.3f}"
)
for idx, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_name:>10} | "
        f"IoU={oi_metrics['iou_per_class'][idx]:.3f} | "
        f"Precision={oi_metrics['precision_per_class'][idx]:.3f} | "
        f"Recall={oi_metrics['recall_per_class'][idx]:.3f} | "
        f"F1={oi_metrics['f1_per_class'][idx]:.3f}"
    )


Necessary images already downloaded
Existing download of split 'validation' is sufficient
Loading existing dataset 'open-images-v7-person-car-dog-seg-100'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use
 100% |█████████████████| 100/100 [2.7s elapsed, 0s remaining, 41.5 samples/s]      
 100% |█████████████████| 100/100 [536.2ms elapsed, 0s remaining, 186.5 samples/s]      
SEGFORMER OPENIMAGES-100 | loss=1.4462 | pixAcc=0.742 | mIoU_fg=0.325
background | IoU=0.706 | Precision=0.972 | Recall=0.721 | F1=0.828
    person | IoU=0.083 | Precision=0.097 | Recall=0.353 | F1=0.153
       car | IoU=0.354 | Precision=0.419 | Recall=0.694 | F1=0.523
       dog | IoU=0.537 | Precision=0.545 | Recall=0.973 | F1=0.699


In [12]:
# Presentation inference: load the saved SegFormer and run on new images
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

MODEL_DIR_TO_LOAD = CHECKPOINT_DIR / "segformer_standalone"
PRESENTATION_IMAGE_PATHS = []
# PRESENTATION_IMAGE_PATHS = [Path(r"C:\path\to\image1.jpg"), Path(r"C:\path\to\image2.jpg")]

inference_processor = SegformerImageProcessor.from_pretrained(MODEL_DIR_TO_LOAD)
inference_model = SegformerForSemanticSegmentation.from_pretrained(MODEL_DIR_TO_LOAD).to(device)
inference_model.eval()

color_map = np.array(
    [
        [0, 0, 0],
        [255, 80, 80],
        [80, 180, 255],
        [80, 220, 120],
    ],
    dtype=np.uint8,
)

@torch.no_grad()
def show_segformer_prediction(image_path: Path):
    img = Image.open(image_path).convert("RGB")
    original_size = img.size
    encoded = inference_processor(images=np.array(img), return_tensors="pt")
    pixel_values = encoded["pixel_values"].to(device)

    outputs = inference_model(pixel_values=pixel_values)
    logits = F.interpolate(
        outputs.logits,
        size=(original_size[1], original_size[0]),
        mode="bilinear",
        align_corners=False,
    )
    pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    pred_rgb = color_map[pred]
    overlay = (0.6 * np.array(img).astype(np.float32) + 0.4 * pred_rgb.astype(np.float32)).clip(0, 255).astype(np.uint8)

    fig, ax = plt.subplots(1, 3, figsize=(14, 5))
    ax[0].imshow(img)
    ax[0].set_title("Image")
    ax[0].axis("off")

    ax[1].imshow(pred, vmin=0, vmax=len(CLASS_NAMES) - 1, cmap="tab10")
    ax[1].set_title("Predicted mask")
    ax[1].axis("off")

    ax[2].imshow(overlay)
    ax[2].set_title("Overlay")
    ax[2].axis("off")

    plt.tight_layout()
    plt.show()
    print("Predicted classes:", [CLASS_NAMES[i] for i in np.unique(pred)])

if not PRESENTATION_IMAGE_PATHS:
    print("Set PRESENTATION_IMAGE_PATHS before running this cell.")
else:
    for image_path in PRESENTATION_IMAGE_PATHS:
        print("Running inference on:", image_path)
        show_segformer_prediction(Path(image_path))


Loading weights: 100%|██████████| 208/208 [00:00<00:00, 1898.73it/s]

Set PRESENTATION_IMAGE_PATHS before running this cell.
